In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import os
from src.utils import load_finetuned_model, load_finetuned_model_lens_from_dir
import pickle
from transformer_lens import HookedTransformer, HookedTransformerConfig


/raid/home/m13521157/absa-eap-ig/enveap/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ['CUDA_VISIBLE_DEVICES'] = '5'  # Set to the appropriate GPU ID

In [3]:
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B", device_map="cuda:5")

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


In [3]:
model_path = 'outputs/models/eap/clean_traintruncated/circuit-eng_finetune-eng/seed_123/aos_sequence_variants/2025-07-22 07:20:55.109610_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20'

In [ ]:
# import torch
# torch.cuda.set_device(5)  # Set default CUDA device to GPU 5

# # ...existing code...
# with open(os.path.join(model_path, 'model_config.pkl'), 'rb') as f:
#     new_cfg_dict = pickle.load(f)
# new_cfg = HookedTransformerConfig.from_dict(new_cfg_dict)
# new_model = HookedTransformer(new_cfg)
# new_model.load_state_dict(torch.load(os.path.join(model_path, 'model.pt'), map_location='cuda'))
# # ...existing code...

In [22]:
# Method 4: Using a more detailed analysis by layer type
def detailed_parameter_count(model):
    layer_counts = {}
    total = 0
    
    for name, param in model.named_parameters():
        layer_type = name.split('.')[0] if '.' in name else name
        param_count = param.numel()
        
        if layer_type not in layer_counts:
            layer_counts[layer_type] = 0
        layer_counts[layer_type] += param_count
        total += param_count
    
    print("Parameter count by layer type:")
    for layer_type, count in layer_counts.items():
        print(f"  {layer_type}: {count:,}")
    print(f"Total: {total:,}")
    
    return total

detailed_parameter_count(model.model.layers)

Parameter count by layer type:
  0: 14,912,384
  1: 14,912,384
  2: 14,912,384
  3: 14,912,384
  4: 14,912,384
  5: 14,912,384
  6: 14,912,384
  7: 14,912,384
  8: 14,912,384
  9: 14,912,384
  10: 14,912,384
  11: 14,912,384
  12: 14,912,384
  13: 14,912,384
  14: 14,912,384
  15: 14,912,384
  16: 14,912,384
  17: 14,912,384
  18: 14,912,384
  19: 14,912,384
  20: 14,912,384
  21: 14,912,384
  22: 14,912,384
  23: 14,912,384
Total: 357,897,216


357897216

In [24]:
def detailed_recursive_count(model, name="model", depth=0, max_depth=3):
    """
    Count parameters with a tree-like structure and size breakdown
    """
    indent = "├─" + "─" * (depth * 2)
    if depth == 0:
        indent = ""
    
    # Count parameters at this level
    total_params = sum(p.numel() for p in model.parameters())
    direct_params = sum(p.numel() for p in model.parameters(recurse=False))
    
    print(f"{indent}{name}: {total_params:,} params")
    
    if depth < max_depth and hasattr(model, 'named_children'):
        children = list(model.named_children())
        for i, (child_name, child_module) in enumerate(children):
            is_last = (i == len(children) - 1)
            child_indent = "└─" if is_last else "├─"
            
            # For the child's children, adjust the prefix
            if depth == 0:
                child_prefix = child_indent
            else:
                child_prefix = indent.replace("├─", "│ ").replace("└─", "  ") + child_indent
            
            child_params = sum(p.numel() for p in child_module.parameters())
            print(f"{child_prefix}{child_name}: {child_params:,} params")
            
            # Recurse if not at max depth
            if depth + 1 < max_depth:
                detailed_recursive_count(child_module, "", depth + 2, max_depth)

# Use it
print("Detailed recursive parameter count:")
print("=" * 60)
detailed_recursive_count(model, "Qwen2.5-0.5B", max_depth=3)

Detailed recursive parameter count:
Qwen2.5-0.5B: 494,032,768 params
├─model: 494,032,768 params
├─────: 494,032,768 params
│ ────├─embed_tokens: 136,134,656 params
│ ────├─layers: 357,897,216 params
│ ────├─norm: 896 params
│ ────└─rotary_emb: 0 params
└─lm_head: 136,134,656 params
├─────: 136,134,656 params


In [19]:
def comprehensive_parameter_analysis(model):
    """
    Comprehensive analysis of model parameters by component type
    """
    component_analysis = {}
    
    def analyze_module(module, path=""):
        module_type = type(module).__name__
        param_count = sum(p.numel() for p in module.parameters(recurse=False))
        
        if module_type not in component_analysis:
            component_analysis[module_type] = {
                'count': 0,
                'total_params': 0,
                'instances': []
            }
        
        if param_count > 0:  # Only count modules with parameters
            component_analysis[module_type]['count'] += 1
            component_analysis[module_type]['total_params'] += param_count
            component_analysis[module_type]['instances'].append({
                'path': path,
                'params': param_count
            })
        
        # Recurse through children
        for name, child in module.named_children():
            child_path = f"{path}.{name}" if path else name
            analyze_module(child, child_path)
    
    analyze_module(model)
    
    print("Component Analysis:")
    print("=" * 80)
    total_params = 0
    
    for component_type, info in sorted(component_analysis.items(), 
                                     key=lambda x: x[1]['total_params'], reverse=True):
        if info['total_params'] > 0:
            print(f"{component_type}:")
            print(f"  Count: {info['count']}")
            print(f"  Total Parameters: {info['total_params']:,}")
            print(f"  Avg Parameters per instance: {info['total_params'] // info['count']:,}")
            total_params += info['total_params']
            print()
    
    print(f"Grand Total: {total_params:,}")
    return component_analysis

# Run the comprehensive analysis
analysis = comprehensive_parameter_analysis(model)

Component Analysis:
Linear:
  Count: 169
  Total Parameters: 493,988,864
  Avg Parameters per instance: 2,923,011

Embedding:
  Count: 1
  Total Parameters: 136,134,656
  Avg Parameters per instance: 136,134,656

Qwen2RMSNorm:
  Count: 49
  Total Parameters: 43,904
  Avg Parameters per instance: 896

Grand Total: 630,167,424


In [21]:
import json
# Save the analysis to a JSON file
output_file = "parameter_analysis.json"
with open(output_file, 'w') as f:
	json.dump(analysis, f, indent=4)